In [1]:
from pathlib import Path
import pandas as pd

input_file = Path(r"/pricesXY.csv")
output_file = Path(r"C:\Users\Carl\Desktop\market_prices_wide.xlsx")

df = pd.read_csv(input_file)
df.columns = df.columns.str.strip()

commodities = [
    "Beans",
    "Cassava (fresh)",
    "Maize (white)",
    "Maize flour",
    "Millet flour",
    "Leafy vegetables",
    "Milk (fresh)"
]

lat_col = "wfp_markets_uga.latitude"
lon_col = "wfp_markets_uga.longitude"

df["Price Date"] = pd.to_datetime(df["Price Date"], dayfirst=True, errors="coerce")
df["Year"] = df["Price Date"].dt.year

df["Price"] = pd.to_numeric(df["Price"], errors="coerce")

df["Commodity"] = df["Commodity"].astype(str).str.strip()
df["Market Name"] = df["Market Name"].astype(str).str.strip()

df = df[df["Commodity"].isin(commodities)].copy()

annual = (
    df
    .groupby(
        ["Market Name", "Year", "Commodity", lat_col, lon_col],
        as_index=False
    )
    .agg(
        Price=("Price", "mean"),
        n_obs=("Price", "count")
    )
)

prices_wide = (
    annual
    .pivot_table(
        index=["Market Name", "Year", lat_col, lon_col],
        columns="Commodity",
        values="Price",
        aggfunc="mean"
    )
    .reset_index()
)

prices_wide.columns.name = None

id_cols = ["Market Name", "Year", lat_col, lon_col]

prices_wide = prices_wide.rename(columns={
    col: (
        col.lower()
        .replace(" ", "_")
        .replace("(", "")
        .replace(")", "")
        + "_price"
    )
    for col in prices_wide.columns
    if col not in id_cols
})

prices_wide = prices_wide.rename(columns={
    lat_col: "Y",
    lon_col: "X"
})

obs_wide = (
    annual
    .pivot_table(
        index=["Market Name", "Year", lat_col, lon_col],
        columns="Commodity",
        values="n_obs",
        aggfunc="sum"
    )
    .reset_index()
)

obs_wide.columns.name = None

obs_wide = obs_wide.rename(columns={
    col: (
        col.lower()
        .replace(" ", "_")
        .replace("(", "")
        .replace(")", "")
        + "_n_obs"
    )
    for col in obs_wide.columns
    if col not in id_cols
})

obs_wide = obs_wide.rename(columns={
    lat_col: "Y",
    lon_col: "X"
})

result = prices_wide.merge(
    obs_wide,
    on=["Market Name", "Year", "X", "Y"],
    how="left"
)

result = result.sort_values(["Market Name", "Year"])

result.to_excel(output_file, index=False)

print("Done.")
print(f"Saved to: {output_file}")
print(result.head())

Done.
Saved to: C:\Users\Carl\Desktop\market_prices_wide.xlsx
                     Market Name  Year     Y      X  beans_price  \
0  Adjumani (refugee settlement)  2020  3.34  31.78  3300.000000   
1  Adjumani (refugee settlement)  2021  3.34  31.78  2945.750000   
2  Adjumani (refugee settlement)  2022  3.34  31.78  3810.807692   
3  Adjumani (refugee settlement)  2023  3.34  31.78  5031.250000   
4  Adjumani (refugee settlement)  2024  3.34  31.78  4541.938967   

   cassava_fresh_price  leafy_vegetables_price  maize_white_price  \
0           627.000000             2052.000000        1037.000000   
1           494.700000             1678.700000        1198.750000   
2           618.100000             1531.100000        1990.621792   
3           746.983333             1879.916667        1845.750000   
4           610.470075             1690.623558        1329.954575   

   maize_flour_price  milk_fresh_price  millet_flour_price  beans_n_obs  \
0        1758.000000       2000.000000 